In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.cuda.amp import GradScaler

# Import utils
from utils.Logger import Logger
from utils.Seed import set_seed
from utils.Splitter import stratified_split
from classes.FeatureDataset.WaveformFeatureDataset import WaveformFeatureDataset
from classes.FeatureDataset.ListDataset import ListDataset

# Import Nes2Net components
from classes.models.Nes2Net.model_Nes2Net import WavLMNes2Net
from classes.models.Nes2Net.trainer_Nes2Net import (
    test_nes2net,
    load_model_nes2net
)

seed = 42
set_seed(42)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ""

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 32
learning_rate = 0.000001
epochs = 30

main_keys = [
    "unseen all samples (UnID)"
]

# WavLM-Nes2Net

## Load Test Data

In [ ]:
# samples will be stored here:
nes2net_test_samples = {
    "unseen all samples (UnID)": [],
}

# unseen spoof datasets
nes2net_spoof_elevenmultilingualv2_converted_dir = "test_preprocessed_data/waveform/Spoof/Converted/ElevenMultilingualV2"
nes2net_spoof_elevenmultilingualv2_tts_dir = "test_preprocessed_data/waveform/Spoof/TTS/ElevenMultilingualV2"
nes2net_spoof_dupdub_converted_dir = "test_preprocessed_data/waveform/Spoof/Converted/DupDub"
nes2net_spoof_dupdub_tts_dir = "test_preprocessed_data/waveform/Spoof/TTS/DupDub"
nes2net_spoof_dupdub_notindataset_tts_dir = "test_preprocessed_data/waveform/Spoof/TTS/DupDub-NotInDataset"

# seen spoof datasets
nes2net_spoof_facebookmms_converted_dir = "preprocessed_data/waveform/Spoof/Converted/FacebookMMS"
nes2net_spoof_facebookmms_tts_dir = "preprocessed_data/waveform/Spoof/TTS/FacebookMMS"
nes2net_spoof_googletts_converted_dir = "preprocessed_data/waveform/Spoof/Converted/GoogleTTS"
nes2net_spoof_googletts_tts_dir = "preprocessed_data/waveform/Spoof/TTS/GoogleTTS"
nes2net_spoof_vits_converted_dir = "preprocessed_data/waveform/Spoof/Converted/VITS"
nes2net_spoof_vits_tts_dir = "preprocessed_data/waveform/Spoof/TTS/VITS"

# Bonafide datasets (split each, DataLoader per split, do not combine)
nes2net_bonafide_commonvoice_dir = "preprocessed_data/waveform/Bonafide/CommonVoice"
nes2net_bonafide_prosa_dir = "preprocessed_data/waveform/Bonafide/Prosa"

## ------------------------------------
## UNSEEN SPOOF DATASETS
## ------------------------------------
# DupDub NotInDataset TTS
if os.path.exists(nes2net_spoof_dupdub_notindataset_tts_dir):
    nes2net_spoof_dupdub_notindataset_tts_dataset = WaveformFeatureDataset(nes2net_spoof_dupdub_notindataset_tts_dir, force_label=0)
    nes2net_spoof_dupdub_notindataset_tts_list = ListDataset([(features, 0) for features, _ in nes2net_spoof_dupdub_notindataset_tts_dataset.samples])
    nes2net_test_samples["unseen all samples (UnID)"].extend(nes2net_spoof_dupdub_notindataset_tts_list)

    print(f"Loaded {len(nes2net_spoof_dupdub_notindataset_tts_list)} samples from {nes2net_spoof_dupdub_notindataset_tts_dir}")
else:
    print(f"Warning: Directory not found: {nes2net_spoof_dupdub_notindataset_tts_dir}")

### Bonafides will be added to "unseen all samples (UID)", "unseen all samples (UnID)", and "seen all samples" so that evaluation metrics can be calculated
# Bonafide CommonVoice
if os.path.exists(nes2net_bonafide_commonvoice_dir):
    dataset_commonvoice = WaveformFeatureDataset(nes2net_bonafide_commonvoice_dir, force_label=1)
    nes2net_bonafide_commonvoice = ListDataset([(features, 1) for features, _ in dataset_commonvoice.samples])
    t_c, v_c, te_c = stratified_split(nes2net_bonafide_commonvoice, splits=(0.7, 0.15, 0.15), seed=seed)

    nes2net_bonafide_commonvoice_list = ListDataset([nes2net_bonafide_commonvoice[i] for i in range(len(te_c))])
    nes2net_test_samples["unseen all samples (UnID)"].extend(nes2net_bonafide_commonvoice_list)

    print(f"Loaded {len(nes2net_bonafide_commonvoice_list)} samples from {nes2net_bonafide_commonvoice_dir}")
else:
    print(f"Warning: Directory not found: {nes2net_bonafide_commonvoice_dir}")

# Bonafide Prosa
if os.path.exists(nes2net_bonafide_prosa_dir):
    dataset_prosa = WaveformFeatureDataset(nes2net_bonafide_prosa_dir, force_label=1)
    nes2net_bonafide_prosa = ListDataset([(features, 1) for features, _ in dataset_prosa.samples])
    t_p, v_p, te_p = stratified_split(nes2net_bonafide_prosa, splits=(0.7, 0.15, 0.15), seed=seed)

    nes2net_bonafide_prosa_list = ListDataset([nes2net_bonafide_prosa[i] for i in range(len(te_p))])
    nes2net_test_samples["unseen all samples (UnID)"].extend(nes2net_bonafide_prosa_list)

    print(f"Loaded {len(nes2net_bonafide_prosa_list)} samples from {nes2net_bonafide_prosa_dir}")
else:
    print(f"Warning: Directory not found: {nes2net_bonafide_prosa_dir}")

print("-----------------------------------------------")
print(f"Total unseen UnID samples: {len(nes2net_test_samples['unseen all samples (UnID)'])}")
print("-----------------------------------------------")

nes2net_test_samples_dataloaders = {
    key: DataLoader(value, batch_size=batch_size, shuffle=False, num_workers=4)
    for key, value in nes2net_test_samples.items()
}

## Test WavLM-Nes2Net Model

In [ ]:
# Model configuration (same as training)
nes2net_model_config = {
    'agg': 'SEA',              # Aggregation method: 'SEA', 'WeightedSum', 'AttM'
    'Nes_ratio': [8, 8],       # Nested Res2Net ratio
    'dilation': 2,             # Dilation factor
    'pool_func': 'mean',       # Pooling function: 'mean' or 'ASTP'
    'SE_ratio': [8],           # SE module ratio
    'cp_path': 'wavlm_large.pt',  # Path to pretrained WavLM model (not used with s3prl)
    'fine_tune_ssl': True      # Whether to fine-tune SSL model
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Set up model, optimizer, and scaler ---
model = WavLMNes2Net(nes2net_model_config, device).to(device)
optimizer = Adam(model.parameters(), lr=learning_rate)
scaler = GradScaler()

# --- Load the trained model ---
# Update the epoch number to the checkpoint you want to test
checkpoint_path = r"pretrained_weights/waveform/Nes2Net/nes2net_waveform-ep_30-bs_32-lr_0.000001.pth"

print(f"\nLoading model from: {checkpoint_path}")
start_epoch = load_model_nes2net(
    model, optimizer, scaler,
    path=checkpoint_path,
    device=device
)

print(f"Loaded checkpoint from epoch {start_epoch}")

# Count parameters
nb_params = sum([param.view(-1).size()[0] for param in model.parameters() if param.requires_grad])
print(f'Number of trainable parameters: {nb_params:,}')

In [ ]:
# Test the model
for key, test_sample in nes2net_test_samples_dataloaders.items():
    print(f"\n{'='*60}")
    print(f"Testing {key}")
    print(f"{'='*60}")
    
    if key in main_keys:
        predictions, targets, metrics = test_nes2net(model, test_sample, device=device)
        
        # Print summary
        print(f"\n--- Results Summary ---")
        print(f"Accuracy: {metrics['accuracy']:.2f}%")
        print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1 Score: {metrics['f1']:.4f}")
        print(f"F2 Score: {metrics['f2']:.4f}")
        print(f"EER: {metrics['eer']:.4f}")
        print(f"actDCF: {metrics['actDCF']:.4f}")
        print(f"minDCF: {metrics['minDCF']:.4f}")
        print(f"CLLR: {metrics['cllr']:.4f}")
    
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Testing completed!")
print(f"{'='*60}")

## Test Different Epochs (Optional)

You can test different epoch checkpoints to find the best performing model.

In [ ]:
# Test multiple epochs
epochs_to_test = [5, 10, 15, 20, 25, 30]  # Adjust as needed

results_summary = []

for epoch in epochs_to_test:
    checkpoint_path = f"pretrained_weights/waveform/Nes2Net/nes2net_waveform-ep_{epoch}-bs_32-lr_0.000001.pth"
    
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        continue
    
    print(f"\n{'='*60}")
    print(f"Testing Epoch {epoch}")
    print(f"{'='*60}")
    
    # Load model
    model = WavLMNes2Net(nes2net_model_config, device).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    scaler = GradScaler()
    
    load_model_nes2net(model, optimizer, scaler, path=checkpoint_path, device=device)
    
    # Test
    for key, test_sample in nes2net_test_samples_dataloaders.items():
        if key in main_keys:
            predictions, targets, metrics = test_nes2net(model, test_sample, device=device)
            
            # Store results
            results_summary.append({
                'epoch': epoch,
                'dataset': key,
                'accuracy': metrics['accuracy'],
                'eer': metrics['eer'],
                'f1': metrics['f1']
            })
            
            print(f"Epoch {epoch} - {key}:")
            print(f"  Accuracy: {metrics['accuracy']:.2f}%, EER: {metrics['eer']:.4f}, F1: {metrics['f1']:.4f}")
    
    torch.cuda.empty_cache()

# Print summary of all epochs
print(f"\n{'='*60}")
print("Summary of All Epochs")
print(f"{'='*60}")
for result in results_summary:
    print(f"Epoch {result['epoch']:2d} | Acc: {result['accuracy']:6.2f}% | EER: {result['eer']:.4f} | F1: {result['f1']:.4f}")